In [ ]:
import pandas as pd
import os
import re
import cv2

In [ ]:
video_path = "/fs3/group/novagrp/Coworkers/Sarah/Locatello/videos/"
annotation_path = "/fs3/group/novagrp/Coworkers/Sarah/Locatello/annotations/"
exp_details_path = "/fs3/group/novagrp/Coworkers/Sarah/Locatello/experimental_conditions.csv"

## Check row data

In [ ]:
# load experimental details
exp_details = pd.read_csv(exp_details_path)

n_rows = exp_details.shape[0]
print(f"Number of rows in experimental conditions: {n_rows}")

# clean columns (remove unnecessary and rename)
exp_details = exp_details.drop(columns=['csv_stage2', 'name', 'V', 'P', 'order'])
exp_details = exp_details.rename(columns={'csv_stage1': 'annotation_file', 
                                          'video': 'observation_file'})
# clean variables values
exp_details['date'] = pd.to_datetime(exp_details['date'], format='%Y%m%d')
exp_details['time'] = exp_details['time'].apply(lambda x: re.sub(r'[-/]', ':', x))
exp_details['time'] = exp_details['time'].apply(lambda x: re.sub(r':(\d{2})(\d{2})$', r':\2', x))
exp_details['time'] = pd.to_datetime(exp_details['time'], format='%H:%M:%S').dt.time
exp_details['observation_file'] = exp_details['observation_file'].apply(lambda x: x.strip() if isinstance(x, str) else x)

display(exp_details.head(3))

In [ ]:
# check notes
exp_details_with_notes = exp_details[exp_details['notes'].notnull()]
n_rows_with_notes = exp_details_with_notes.shape[0]
print(f"Number of rows with notes: {n_rows_with_notes}")
display(exp_details_with_notes)

In [ ]:
# ignore from exp_details all seed=7
exp_details = exp_details[exp_details['seed'] != 7]
n_rows_after_seed7 = exp_details.shape[0]
print(f"Number of rows after removing seed=7: {n_rows_after_seed7}")
display(exp_details.head(4))

In [ ]:
# check seeds
pool_counts = exp_details['pool'].value_counts()
print("Number of rows per pool value:")
print(pool_counts)

# check errors: e.g., rd57_3
rd57_3_details = exp_details[exp_details['pool'] == 'rd57_3']
n_rd57_3 = rd57_3_details.shape[0]
print(f"Number of rows in pool rd57_3: {n_rd57_3}")
display(rd57_3_details)

In [ ]:
# check if any observation_file is missing in video_path and list the missing ones
missing_videos = []
for video_file in exp_details['observation_file']:
    if not os.path.isfile(os.path.join(video_path, video_file)):
        missing_videos.append(video_file)
if len(missing_videos) > 0:
    print("Missing video files:")
    for mv in missing_videos:
        print(mv)
else:
    print("All video files are present.")


In [ ]:
# for each row in exp_details,check video lentgth (in frames), fps and video resolution and add corresponding columns
video_lengths = []
video_fps = []
video_resolutions = []
for video_file in exp_details['observation_file']:
    video_path_full = os.path.join(video_path, video_file)
    cap = cv2.VideoCapture(video_path_full)
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    resolution = f"{width}x{height}"
    video_lengths.append(length)
    video_fps.append(fps)
    video_resolutions.append(resolution)
    cap.release()
exp_details['video_length_frames'] = video_lengths
exp_details['video_fps'] = video_fps
exp_details['video_resolution'] = video_resolutions
display(exp_details.head(6))

# print their unique values and occurrences
print("Video lengths (frames) unique values and occurrences:")
print(exp_details['video_length_frames'].value_counts())
print("Video FPS unique values and occurrences:")
print(exp_details['video_fps'].value_counts())
print("Video resolutions unique values and occurrences:")
print(exp_details['video_resolution'].value_counts())

In [ ]:
# length distribution
video_length_counts = exp_details['video_length_frames'].value_counts()
print("All video_length_frames values and occurrences:")
for length, count in video_length_counts.items():
    print(f"Length: {length}, Count: {count}")
short_videos = exp_details[exp_details['video_length_frames'] < 5000]
n_short_videos = short_videos.shape[0]
print(f"\nNumber of videos with length < 5000 frames: {n_short_videos}")
display(short_videos)

In [ ]:
# fps distribution
videos_25fps = exp_details[exp_details['video_fps'] == 25.0]
n_videos_25fps = videos_25fps.shape[0]
print(f"Number of videos with no FPS: {n_videos_25fps}")
display(videos_25fps)
exp_details_filtered = exp_details[exp_details['video_fps'] != 25.0]
n_rows_filtered = exp_details_filtered.shape[0]
print(f"Number of rows after removing 25 FPS videos: {n_rows_filtered}")
display(exp_details_filtered.head(10))

In [ ]:
# check notes
exp_details_with_notes = exp_details[exp_details['notes'].notnull()]
n_rows_with_notes = exp_details_with_notes.shape[0]
print(f"Number of rows with notes: {n_rows_with_notes}")
display(exp_details_with_notes)

# remove notes column
exp_details = exp_details.drop(columns=['notes', 'video_length_frames', 'video_fps', 'video_resolution'])

In [ ]:
# get a extra column called 'valid' always equal to 1, except for pool rd33
exp_details['valid'] = 1
exp_details.loc[exp_details['pool'] == 'rd33', 'valid'] = 0

# add missing columns
# add observation_id column as {genotype}_{line}_{sex}_{seed}_{odor}_{phase}
exp_details['observation_id'] = exp_details.apply(lambda row: f"{row['genotype']}_{row['line']}_{row['sex']}_{row['seed']}_{row['odor']}_{row['phase']}", axis=1)
# add start_frame=0 column
exp_details['start_frame'] = 0
# add end_frame column = 54000 if phase='H', elif for 'O' or 'P' 27000
exp_details['end_frame'] = exp_details.apply(lambda row: 54000 if row['phase'] == 'H' else 27000, axis=1)
exp_details

## Get source data

In [ ]:
# v1 genotype not mixed and v2 genotype mixed
exp_details_v1 = exp_details[exp_details['genotype'] != 'mixed']
exp_details_v1.to_csv('/nfs/scistore19/locatgrp/rcadei/artificial-causal-inference/data/mice/v1/experiment.csv', index=False)
exp_details_v2 = exp_details[exp_details['genotype'] == 'mixed']
exp_details_v2.to_csv('/nfs/scistore19/locatgrp/rcadei/artificial-causal-inference/data/mice/v2/experiment.csv', index=False)